In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about various aspects of Lunapolis, the capital of the moon, including its weather and demographics, and is interested in the cheese miners' union's potential actions.\n\n## SUMMARY\nThe capital of the moon is Lunapolis. The weather in Lunapolis is characterized by clear skies, with temperatures ranging from a high of 120°C to a low of -100°C. There are 100,000 cheese miners living in Lunapolis, and it is anticipated that the cheese miners' union may go on strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='ca1eee72-48f7-46ef-b071-b0e43f19d6ad'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='c68246d0-c2ed-4030-b4

In [8]:
print(response["messages"][0].content)


Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various aspects of Lunapolis, the capital of the moon, including its weather and demographics, and is interested in the cheese miners' union's potential actions.

## SUMMARY
The capital of the moon is Lunapolis. The weather in Lunapolis is characterized by clear skies, with temperatures ranging from a high of 120°C to a low of -100°C. There are 100,000 cheese miners living in Lunapolis, and it is anticipated that the cheese miners' union may go on strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [11]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [12]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='90b1fa63-471d-45f8-bb53-2a5664c6d848'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='0e03f7f3-aa0b-4de2-bd05-ec834259f964', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='cf9215dd-831b-41ca-b919-cb6fb1da750f'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='de5f99fa-44ba-469f-8434-4b3ed4d325e6', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='3dee80e9-2b45-4b12-a14f-c43fedada717'),
              AIMessage(content='I can’t read your device’s temperature from here. If you want 

In [13]:
print(response["messages"][-1].content)

I can’t read your device’s temperature from here. If you want to check it yourself, tell me what kind of device it is (e.g., Windows PC, Mac, Android phone, iPhone, Linux laptop, router, etc.) and I’ll give you exact steps. In the meantime, here are quick ways by device type:

- Windows PC
  - Use a hardware monitoring tool (HWMonitor, HWiNFO64, Core Temp) to see CPU/GPU temps.
  - You can also check the BIOS/UEFI hardware monitor on startup for temperatures.

- Mac
  - Macs don’t show temps in Activity Monitor. Install a third-party tool like iStat Menus or Macs Fan Control to read CPU temps and fan speeds.

- Linux
  - Install lm-sensors (sudo apt install lm-sensors; sudo sensors-detect) and run sensors to view core temps.

- Android
  - Use apps like CPU-Z or AIDA64 to read CPU/GPU temperatures.

- iPhone
  - iOS doesn’t expose temps in a simple UI. If it’s heating during charging, try removing case, charging in a cool area, and check battery health in Settings > Battery > Battery H